## 1. Encabezado e imports
Ahora vamos a importar todas las librerías que usaremos en el cuaderno, en una sola celda.

In [ ]:
import random  # semilla y utilidades de azar de Python
from pathlib import Path # manejo de rutas
import shutil  # copiar los .npz de Drive a /content
import numpy as np  # arreglos y carga de los .npz
import pandas as pd  # lectura de splits.csv
import matplotlib.pyplot as plt  # gráficas e inspección de imágenes
import cv2  # lectura y redimensionado de imágenes
import tensorflow as tf  # verificación de GPU
import keras  # construcción y entrenamiento de la red
from keras import layers  # capas del modelo y del bloque de aumento
from google.colab import drive  # montar Drive
import json
import hashlib
import zipfile
from sklearn.utils.class_weight import compute_class_weight  # pesos de clase balanceados
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay  # métricas de evaluación


## 2. Configuración y entorno
Ahora vamos a definir las constantes comunes del proyecto, verificar la GPU, montar Drive y fijar la semilla.

In [ ]:
SEED = 42 # semilla para que la distrubución de datos sea la misma en todo el proyecto
random.seed(SEED)
np.random.seed(SEED)
# semilla también para Keras
keras.utils.set_random_seed(SEED)
EPOCHS = 60

# hiperparámetro de entrenamiento
BATCH_SIZE = 32

# rutas en Drive (persisten)
DRIVE_ROOT = Path("/content/drive/MyDrive/sylfish")
ZIP_DRIVE = DRIVE_ROOT / "data" / "raw" / "SylFishBD_IMAGE.zip"
SPLITS_DIR = DRIVE_ROOT / "data" / "splits"
PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"
MASKS_ZIP = DRIVE_ROOT / "data" / "masks.zip"
TABLES_DIR = DRIVE_ROOT / "reports" / "tables"
FIGURES_DIR = DRIVE_ROOT / "reports" / "figures"

# rutas locales (se borran al reiniciar la sesión)
LOCAL_ROOT = Path("/content")
EXTRACT_DIR = LOCAL_ROOT / "sylfishbd"
MASKS_DIR = LOCAL_ROOT / "masks"

# rutas de salida (modelo entrenado y comparación DL vs ML)
MODELS_DIR = DRIVE_ROOT / "models"
COMPARACION_DIR = DRIVE_ROOT / "comparacion"
LOCAL_MODELS = Path("/content/models")   # disco local de Colab: rápido y sin duplicados
LOCAL_MODELS.mkdir(parents=True, exist_ok=True)

# clases en el orden de classes.txt (el índice es el label), es decir nuestras clases serán los numeros del 0 al 8 en el mismo orden del txt
# 0 - boal , 1 - ilish, ..., 8 - telapia
CLASES = ["boal", "ilish", "kalibaush", "katla", "koi",
          "mrigel", "pabda", "rui", "telapia"]
NUM_CLASES = len(CLASES)
CLASE_A_LABEL = {}
for i, c in enumerate(CLASES):
    CLASE_A_LABEL[c] = i
print(f"Clases ({NUM_CLASES}):", CLASE_A_LABEL)

# tamaños de images e interpolaciones para bajar carga computacional
IMG_CNN = 224
IMG_UNET = 256
INTER_IMG = cv2.INTER_AREA        # imágenes: promedia píxeles al reducir
INTER_MASK = cv2.INTER_NEAREST    # máscaras: no inventa grises

# particion y umbral por defecto
FRAC_TRAIN, FRAC_VAL, FRAC_TEST = 0.70, 0.15, 0.15
UMBRAL_MASCARA = 0.5

In [ ]:
drive.mount('/content/drive')

## 3. Carga del caché
Ahora vamos a copiar los .npz de train y val desde Drive a /content y abrirlos. El de test también se carga aquí, pero no se toca hasta la sección final.

In [ ]:
def cargar_particion(split):
    datos = np.load(PROCESSED_DIR / f"cnn_224_{split}.npz", allow_pickle=True)  # leemos el .npz de Drive
    return datos["X"], datos["y"], datos["filepath"]

X_train, y_train, fp_train = cargar_particion("train")
X_val, y_val, fp_val = cargar_particion("val")
X_test, y_test, fp_test = cargar_particion("test")  # solo se usa al final, una vez

# verificación: formas, tipo y rango de etiquetas
for nombre, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(nombre, X.shape, X.dtype, y.shape, y.min(), y.max())

# verificación: ninguna imagen aparece en dos particiones (sin fuga)
print(len(set(fp_train) & set(fp_val)), len(set(fp_train) & set(fp_test)), len(set(fp_val) & set(fp_test)))

Ahora revisamos que lo que obtuvimos concuerde con lo que se encuentra en `splits.csv` sea lo mismo

In [ ]:
splits_df = pd.read_csv(SPLITS_DIR / "splits.csv")           # tabla oficial de la partición
print("Valores de split en el CSV:", splits_df["split"].unique())

def _norm(p):
    return str(p).replace("\\", "/")                          # unifica separadores de ruta

for nombre, y, fp in [("train", y_train, fp_train),
                      ("val", y_val, fp_val),
                      ("test", y_test, fp_test)]:
    sub = splits_df[splits_df["split"] == nombre].reset_index(drop=True)
    print(f"{nombre}: npz={len(y)} | splits.csv={len(sub)}")

    # 1) mismo número de filas
    assert len(sub) == len(y), f"{nombre}: {len(sub)} filas en CSV vs {len(y)} en npz"

    # 2) mismo conteo por clase
    cuenta_npz = np.bincount(y, minlength=NUM_CLASES)
    cuenta_csv = np.bincount(sub["label"].to_numpy(), minlength=NUM_CLASES)
    assert (cuenta_npz == cuenta_csv).all(), f"{nombre}: conteo por clase distinto\n{cuenta_npz}\n{cuenta_csv}"

    # 3) mismo orden de filepath
    fp_npz = [_norm(p) for p in fp]
    fp_csv = [_norm(p) for p in sub["filepath"]]
    if fp_npz != fp_csv:
        print("  ejemplo npz:", fp_npz[:2])
        print("  ejemplo csv:", fp_csv[:2])
        print("  mismo conjunto de rutas:", set(fp_npz) == set(fp_csv))
    assert fp_npz == fp_csv, f"{nombre}: filepath distinto o en otro orden"

    # 4) mismo label fila a fila
    assert (sub["label"].to_numpy() == y).all(), f"{nombre}: label no coincide fila a fila"

print("Verificación contra splits.csv: OK")

## 4. Inspección rápida

Ahora vamos a revisar visualmente el caché de train: una imagen por clase, con el nombre de clase y el nombre del archivo en el título. Queremos confirmar tres cosas antes de seguir: que el color esté en RGB (los peces no deben verse azulados) y que la etiqueta numérica corresponda a la clase de `classes.txt`


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(9, 9))  # grilla 3x3
axes = axes.ravel()  # aplanamos los ejes para iterar

for i, clase in enumerate(CLASES):
    idx = np.where(y_train == i)[0][0]  # primera de la clase en train
    img = X_train[idx]  # imagen uint8 en [0, 255]
    nombre_archivo = Path(fp_train[idx]).name  # solo el nombre, sin la ruta completa
    axes[i].imshow(img)
    axes[i].set_title(f"{clase}\n{nombre_archivo}", fontsize=9)
    axes[i].axis("off") # sin ejes, la imagen manda

plt.tight_layout()  # separa los títulos para que no se encimen
plt.show()

## 5. Pesos de clase

Ahora vamos a calcular un peso por clase para compensar el desbalance de train. La fórmula es $w_c = N / (K \cdot N_c)$, con $N$ el total de imágenes, $K=9$ clases y $N_c$ las imágenes de esa clase. Sin pesos, el modelo favorece las clases grandes (rui, mrigel, telapia) y el F1 macro cae. Con pesos, los errores en clases pequeñas pesan más en la pérdida.


In [ ]:
pesos = compute_class_weight(
    class_weight="balanced",           # fórmula N / (K * Nc) por clase
    classes=np.arange(NUM_CLASES),     # labels 0..8 en orden del contrato
    y=y_train,                         # SOLO train, nunca val ni test
)

class_weight = dict(zip(np.arange(NUM_CLASES), pesos))

for label, peso in sorted(class_weight.items(), key=lambda kv: -kv[1]):
    print(f"{CLASES[label]:12s} (label {label}): {peso:.5}")

## 6. Bloque de aumento (Data Augmentation)

Ahora vamos a definir el bloque de aumento que irá dentro del modelo y a verificar visualmente que las transformaciones son razonables para peces. Las fotos son cenitales sobre superficie lisa, así que la orientación del pez es arbitraria: flips y rotación no cambian la especie. La distancia de la cámara varía (zoom) y la iluminación cambia entre tomas (contraste).

In [ ]:
def construir_aumento(seed=None):
    return keras.Sequential(
        [
            keras.Input(shape=(IMG_CNN, IMG_CNN, 3)),
            layers.RandomFlip("horizontal_and_vertical", seed=seed),
            layers.RandomRotation(0.1, fill_mode="reflect", seed=seed),
            layers.RandomZoom(0.1, seed=seed),
            layers.RandomContrast(0.1, seed=seed),
        ],
        name="aumento",
    )

aumento = construir_aumento(seed=SEED)  # bloque que usa el paso 7

In [ ]:
img = X_train[0]                                      # una imagen uint8 de train
img_batch = tf.expand_dims(img, axis=0)               # agrega batch: (1, 224, 224, 3)

tf.random.set_seed(SEED)                              # semilla local para reproducibilidad de la grilla

fig, axes = plt.subplots(3, 3, figsize=(9, 9))        # 3x3: original + 8 aumentadas
axes = axes.ravel()                                   # aplanamos para indexar con un solo número

axes[0].imshow(img)                                   # la original
axes[0].set_title("Original", fontsize=10)
axes[0].axis("off")

aumento = construir_aumento(seed=SEED)
for k in range(8):
    aug = aumento(img_batch, training=True)[0].numpy()# training=True activa las Random
    aug = np.clip(aug, 0, 255).astype(np.uint8)       # por si alguna capa devuelve float
    axes[k + 1].imshow(aug)                           # k+1 porque la celda 0 es la original
    axes[k + 1].set_title(f"Aumentada {k + 1}", fontsize=10)
    axes[k + 1].axis("off")

plt.tight_layout()
plt.show()

## 7. Modelo

Ahora vamos a construir la CNN que clasifica las 9 especies.


In [ ]:
entrada = keras.Input(shape=(IMG_CNN, IMG_CNN, 3), name="entrada")  # (None, 224, 224, 3)
x = aumento(entrada)
x = layers.Rescaling(1.0 / 255, name="rescalado")(x)                 # [0, 1] float32

for i, filtros in enumerate([32, 64, 128, 256, 256], start=1):
    x = layers.Conv2D(filtros, (3, 3), padding="same", name=f"conv_{i}_{filtros}")(x)
    x = layers.BatchNormalization(name=f"bn_{i}")(x)
    x = layers.ReLU(name=f"relu_{i}")(x)
    x = layers.MaxPooling2D((2, 2), name=f"pool_{i}")(x)

x = layers.SpatialDropout2D(0.3, name="spatial_dropout")(x)         # apaga canales completos
x = layers.GlobalAveragePooling2D(name="gap")(x)                     # (None, 7, 7, 256) → (None, 256)
x = layers.Dense(256, name="dense_256")(x)
x = layers.BatchNormalization(name="bn_densa")(x)
x = layers.ReLU(name="relu_densa")(x)
x = layers.Dropout(0.2, name="dropout_densa")(x)
salida = layers.Dense(NUM_CLASES, activation="softmax", name="salida")(x)

modelo = keras.Model(entrada, salida, name="cnn_sylfish")

modelo.summary()

print()
print("=== Verificación ===")
print(f"Salida del modelo: {modelo.output.shape}")                     # esperado (None, 9)
print(f"Mapa antes del GAP (pool_5): {modelo.get_layer('pool_5').output.shape}")  # esperado (None, 7, 7, 256)
print(f"Params totales: {modelo.count_params():,}")                     # esperado 1,050,569

## 8. Compilación

Ahora vamos a compilar el modelo: elegir optimizador, función de pérdida y métrica. No entrena todavía

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 9. Callbacks

Ahora vamos a definir los 5 callbacks que controlan el entrenamiento. No entrenan nada aquí; se pasan a `model.fit(..., callbacks=[...])` en el paso 10.


In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)   # crea MyDrive/sylfish/models si no existe
TABLES_DIR.mkdir(parents=True, exist_ok=True)   # crea reports/tables si no existe

EXPERIMENTO = "linea_base"                       # etiqueta del CSV y de la tabla de experimentos

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",                       # se detiene sobre la pérdida de val
        patience=8,                               # tolera 8 épocas sin mejorar
        restore_best_weights=True,                # al detenerse, deja los pesos del mejor val_loss
        mode="min",                               # se minimiza val_loss
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "cnn_best.keras"),  # guarda en Drive
        monitor="val_loss",
        save_best_only=True,                      # solo guarda si val_loss mejora
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "cnn_last.keras"),  # se sobrescribe cada época
        save_best_only=False,                     # siempre guarda
        verbose=0,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,                               # lr *= 0.5
        patience=3,                               # tolera 3 épocas sin mejorar
        mode="min",
        verbose=1,
    ),
    keras.callbacks.CSVLogger(
        filename=str(TABLES_DIR / f"cnn_historial_{EXPERIMENTO}.csv"),
    ),
]

for cb in callbacks:
    print(type(cb).__name__, getattr(cb, "monitor", "—"))

## 10. Entrenamiento de línea base

Ahora vamos a entrenar el modelo con las imágenes de train y a monitorear la pérdida y accuracy en val. El `fit` corre hasta `EPOCHS=60`, pero el `EarlyStopping` del paso 9 puede cortarlo antes si `val_loss` deja de mejorar. El `ReduceLROnPlateau` baja el lr ×0.5 si `val_loss` se estanca 3 épocas.

In [ ]:
def cargar_modelos(id_exp="linea_base", cambio=None, notas="", tol=1e-3):

    sufijo = "" if id_exp == "linea_base" else f"_{id_exp}"   # la línea base no lleva sufijo
    ruta_best = MODELS_DIR / f"cnn_best{sufijo}.keras"
    ruta_last = MODELS_DIR / f"cnn_last{sufijo}.keras"
    ruta_csv = TABLES_DIR / f"cnn_historial_{id_exp}.csv"

    if not (ruta_best.exists() and ruta_last.exists() and ruta_csv.exists()):
        print(f"[{id_exp}] faltan archivos en Drive; hay que entrenar.")
        return None

    modelo_best = keras.models.load_model(ruta_best)
    modelo_last = keras.models.load_model(ruta_last)

    # verificación contra el historial, por si Drive entregó una copia vieja (duplicados)
    hist_df = pd.read_csv(ruta_csv)
    mejor_epoca = int(hist_df.loc[hist_df["val_loss"].idxmin(), "epoch"])
    esperado_best = hist_df["val_loss"].min()
    esperado_last = hist_df["val_loss"].iloc[-1]

    loss_best = modelo_best.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)[0]
    loss_last = modelo_last.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)[0]

    for nombre, obtenido, esperado in [("best", loss_best, esperado_best),
                                       ("last", loss_last, esperado_last)]:
        ok = abs(obtenido - esperado) <= tol
        estado = "OK" if ok else "NO COINCIDE (posible copia vieja en Drive)"
        print(f"[{id_exp}] {nombre}: val_loss={obtenido:.4f} | historial={esperado:.4f} -> {estado}")

    metrics = None
    if cambio is not None:                                    # modo experimento: evalúa y registra
        _, _, metrics = evaluar(modelo_best, X_val, y_val)
        registrar_experimento(id_exp, cambio, metrics, mejor_epoca, notas)

    return modelo_best, modelo_last, mejor_epoca, metrics

In [ ]:
hist = modelo.fit(
    X_train, y_train,                        # entrenamiento: features y etiquetas enteras
    validation_data=(X_val, y_val),          # monitoreo: NUNCA test
    class_weight=class_weight,               # dict del paso 5, compensa el desbalance
    batch_size=BATCH_SIZE,                   # 32, definido en la sección 2
    epochs=EPOCHS,                           # 60, definido en la sección 2
    callbacks=callbacks,                     # lista del paso 9
    verbose=1,                               # barra de progreso por época
)

In [ ]:
resultado = cargar_modelos("linea_base")
if resultado is None:
    print("No hay modelo guardado: corre el fit de esta sección.")
else:
    modelo_best, modelo_last, mejor_epoca, _ = resultado
    modelo = modelo_best      # las secciones siguientes usan `modelo`

Verificacion


In [ ]:

print(f"Checkpoint best existe: {(MODELS_DIR / 'cnn_best.keras').exists()}")
print(f"Checkpoint last existe: {(MODELS_DIR / 'cnn_last.keras').exists()}")
print(f"CSV historial existe: {(TABLES_DIR / f'cnn_historial_{EXPERIMENTO}.csv').exists()}")

## 11. Curvas

Ahora vamos a leer el CSV que dejó el `CSVLogger` y graficar la pérdida y la accuracy por época, train contra val. La figura se guarda en `reports/figures/` como evidencia del comportamiento del entrenamiento.


In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)   # crea reports/figures si no existe

hist_df = pd.read_csv(TABLES_DIR / f"cnn_historial_{EXPERIMENTO}.csv")

mejor_epoca = int(hist_df.loc[hist_df["val_loss"].idxmin(), "epoch"])  # época del mejor val_loss
epoca_final = int(hist_df["epoch"].max())                              # última época registrada

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

# --- subgráfica izquierda: pérdida ---
ax_loss.plot(hist_df["epoch"], hist_df["loss"], label="train loss", color="tab:blue")
ax_loss.plot(hist_df["epoch"], hist_df["val_loss"], label="val loss", color="tab:orange")
ax_loss.axvline(mejor_epoca, color="green", linestyle="--", alpha=0.7, label=f"mejor val_loss (ép. {mejor_epoca})")
ax_loss.axvline(epoca_final, color="red", linestyle=":", alpha=0.7, label=f"early stopping (ép. {epoca_final})")
ax_loss.set_xlabel("Época")
ax_loss.set_ylabel("Pérdida (CE)")
ax_loss.set_title(f"Pérdida — {EXPERIMENTO}")
ax_loss.set_ylim(0, hist_df["val_loss"].max() * 1.1)   # acota al máximo de val_loss
ax_loss.legend()
ax_loss.grid(alpha=0.3)

# --- subgráfica derecha: accuracy ---
ax_acc.plot(hist_df["epoch"], hist_df["accuracy"], label="train acc", color="tab:blue")
ax_acc.plot(hist_df["epoch"], hist_df["val_accuracy"], label="val acc", color="tab:orange")
ax_acc.axvline(mejor_epoca, color="green", linestyle="--", alpha=0.7, label=f"mejor val_loss (ép. {mejor_epoca})")
ax_acc.axvline(epoca_final, color="red", linestyle=":", alpha=0.7, label=f"early stopping (ép. {epoca_final})")
ax_acc.set_xlabel("Época")
ax_acc.set_ylabel("Accuracy")
ax_acc.set_title(f"Accuracy — {EXPERIMENTO}")
ax_acc.set_ylim(0, 1)                                  # accuracy siempre entre 0 y 1
ax_acc.legend()
ax_acc.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f"cnn_curvas_{EXPERIMENTO}.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Época del mejor val_loss: {mejor_epoca}")
print(f"Época del early stopping: {epoca_final}")
print(f"Figura guardada en: {FIGURES_DIR / f'cnn_curvas_{EXPERIMENTO}.png'}")

## 12. Evaluación alineada en val

Ahora vamos a definir una función `evaluar(modelo, X, y)` la idea es tener un único lugar donde se calculan accuracy, F1 macro, `classification_report` y matriz de confusión normalizada, para que todos los experimentos usen exactamente la misma rutina.

In [ ]:
def evaluar(modelo, X, y):

    probs = modelo.predict(X, verbose=0)                       # predicción completa, sin shuffle
    preds = np.argmax(probs, axis=1)                          # clase más probable por fila

    assert len(preds) == len(y), f"Desalineado: {len(preds)} predicciones vs {len(y)} etiquetas"

    acc = accuracy_score(y, preds)
    f1_macro = f1_score(y, preds, average="macro", zero_division=0)
    report_dict = classification_report(
        y, preds,
        target_names=CLASES,
        digits=3,
        output_dict=True,
        zero_division=0,
    )
    cm_normalized = confusion_matrix(y, preds, normalize="true")

    # verificar que la matriz normalizada está bien formada
    assert cm_normalized.min() >= 0 and cm_normalized.max() <= 1
    assert np.allclose(cm_normalized.sum(axis=1), 1)

    metrics_dict = {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "report_dict": report_dict,
        "cm_normalized": cm_normalized,
    }
    return probs, preds, metrics_dict


# --- aplicar a val ---
probs_val, preds_val, metrics_val = evaluar(modelo, X_val, y_val)

print(f"Accuracy  (val): {metrics_val['accuracy']:.4f}")
print(f"F1 macro  (val): {metrics_val['f1_macro']:.4f}")
print()
print(classification_report(
    y_val, preds_val,
    target_names=CLASES,
    digits=3,
    zero_division=0,
))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=metrics_val["cm_normalized"],
    display_labels=CLASES,
)
disp.plot(
    ax=ax,
    cmap="Blues",
    values_format=".2f",
    colorbar=False,
)
ax.set_title(f"Matriz de confusión normalizada — val ({EXPERIMENTO})")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"cnn_matriz_val_{EXPERIMENTO}.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Registro del experimento

Ahora vamos a registrar la línea base en `reports/tables/cnn_experimentos.csv`.

In [ ]:
def registrar_experimento(id_exp, cambio, metrics, mejor_epoca, notas=""):

    COLS = ["id", "cambio", "accuracy_val", "f1_macro_val", "mejor_epoca", "notas"]
    ruta = TABLES_DIR / "cnn_experimentos.csv"

    if ruta.exists():
        df = pd.read_csv(ruta)
        df = df[df["id"] != id_exp]                          # quita la fila previa si existe
    else:
        df = pd.DataFrame(columns=COLS)                       # CSV nuevo con las columnas

    nueva = pd.DataFrame(
        [{
            "id": id_exp,
            "cambio": cambio,
            "accuracy_val": round(float(metrics["accuracy"]), 4),
            "f1_macro_val": round(float(metrics["f1_macro"]), 4),
            "mejor_epoca": int(mejor_epoca),
            "notas": notas,
        }],
        columns=COLS,
    )

    df = pd.concat([df, nueva], ignore_index=True)
    df.to_csv(ruta, index=False)
    return df


df_exp = registrar_experimento(
    id_exp="linea_base",
    cambio="punto de partida (línea base)",
    metrics=metrics_val,
    mejor_epoca=mejor_epoca,
    notas="lr=1e-3, ES patience=8, LR patience=3, batch=32",
)

print(df_exp.to_string(index=False))
print()
print(f"Guardado en: {TABLES_DIR / 'cnn_experimentos.csv'}")

## 14. Fase de mejoras

Ahora vamos a probar tres variantes de la línea base, **una variable por experimento**, con la misma partición, semilla y presupuesto de épocas:

- **`l2`**: `L2(1e-4)` en las 5 `Conv2D` y en `Dense(256)`.
- **`aumento`**: rotación 0.1 → 0.25 y `RandomErasing(0.1)`
- **`sin_cw`**: quitar `class_weight`.

### 14.1. Función `construir_modelo`

Ahora vamos a definir la función que arma un modelo fresco con la arquitectura del paso 7 y variantes opcionales: `rot` (factor de rotación), `erasing` (agregar `RandomErasing`) y `l2` (regularización L2 en convs y densa).

In [ ]:
def construir_modelo(rot=0.1, erasing=False, l2=None):

    capas_aug = [
        keras.Input(shape=(IMG_CNN, IMG_CNN, 3)),
        layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        layers.RandomRotation(rot, fill_mode="reflect", seed=SEED),
        layers.RandomZoom(0.1, seed=SEED),
        layers.RandomContrast(0.1, seed=SEED),
    ]
    if erasing:
        capas_aug.append(layers.RandomErasing(0.1, seed=SEED))
    aumento_local = keras.Sequential(capas_aug, name="aumento")

    reg = keras.regularizers.L2(l2) if l2 is not None else None

    entrada = keras.Input(shape=(IMG_CNN, IMG_CNN, 3), name="entrada")
    x = aumento_local(entrada)
    x = layers.Rescaling(1.0 / 255, name="rescalado")(x)

    for i, filtros in enumerate([32, 64, 128, 256, 256], start=1):
        x = layers.Conv2D(filtros, (3, 3), padding="same",
                          kernel_regularizer=reg, name=f"conv_{i}_{filtros}")(x)
        x = layers.BatchNormalization(name=f"bn_{i}")(x)
        x = layers.ReLU(name=f"relu_{i}")(x)
        x = layers.MaxPooling2D((2, 2), name=f"pool_{i}")(x)

    x = layers.SpatialDropout2D(0.3, name="spatial_dropout")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, kernel_regularizer=reg, name="dense_256")(x)
    x = layers.BatchNormalization(name="bn_densa")(x)
    x = layers.ReLU(name="relu_densa")(x)
    x = layers.Dropout(0.2, name="dropout_densa")(x)
    salida = layers.Dense(NUM_CLASES, activation="softmax", name="salida")(x)

    return keras.Model(entrada, salida, name="cnn_sylfish")

### 14.2. Función `construir_callbacks`

Ahora vamos a definir la función que arma la lista de callbacks del paso 9 con los nombres de archivo ajustados por `id_exp`. Cada experimento tiene su propio `cnn_best_<id>.keras`, `cnn_last_<id>.keras` y `cnn_historial_<id>.csv`

In [ ]:
def construir_callbacks(id_exp):

    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True,
            mode="min", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(MODELS_DIR / f"cnn_best_{id_exp}.keras"),
            monitor="val_loss", save_best_only=True, mode="min", verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(MODELS_DIR / f"cnn_last_{id_exp}.keras"),
            save_best_only=False, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, mode="min", verbose=1,
        ),
        keras.callbacks.CSVLogger(
            filename=str(TABLES_DIR / f"cnn_historial_{id_exp}.csv"),
        ),
    ]

### 14.3. Función `correr_experimento`

Ahora vamos a definir la función que orquesta un experimento completo: fija la semilla, construye el modelo con las variantes que se le pasen, compila, entrena con los callbacks del experimento, evalúa en val con la rutina del paso 12

In [ ]:
def correr_experimento(id_exp, cambio, notas="", lr=1e-3, use_class_weight=True, **kwargs_modelo):
    keras.utils.set_random_seed(SEED)

    modelo_exp = construir_modelo(**kwargs_modelo)
    modelo_exp.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    modelo_exp.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        class_weight=class_weight if use_class_weight else None,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=construir_callbacks(id_exp),
        verbose=1,
    )

    hist_exp = pd.read_csv(TABLES_DIR / f"cnn_historial_{id_exp}.csv")
    mejor_epoca = int(hist_exp.loc[hist_exp["val_loss"].idxmin(), "epoch"])

    _, _, metrics_exp = evaluar(modelo_exp, X_val, y_val)
    registrar_experimento(id_exp, cambio, metrics_exp, mejor_epoca, notas)

    return modelo_exp, metrics_exp

### 14.4. Experimento `l2`

In [ ]:
modelo_l2, metrics_l2 = correr_experimento(
    id_exp="l2",
    cambio="L2(1e-4) en convs y Dense(256)",
    notas="regularización L2, resto igual a línea base",
    l2=1e-4,
)

In [ ]:
resultado = cargar_modelos(
    "l2",
    cambio="L2(1e-4) en convs y Dense(256)",
    notas="regularización L2, resto igual a línea base",
)
if resultado is None:
    print("No hay modelo guardado: corre el fit de esta sección.")
else:
    modelo_l2, _, _, metrics_l2 = resultado

### 14.5. Experimento `aumento`

In [ ]:
modelo_aumento, metrics_aumento = correr_experimento(
    id_exp="aumento",
    cambio="RandomRotation 0.1→0.25 + RandomErasing(0.1)",
    notas="aumento más agresivo, resto igual a línea base",
    rot=0.25,
    erasing=True,
)

In [ ]:
resultado = cargar_modelos(
    "aumento",
    cambio="RandomRotation 0.1→0.25 + RandomErasing(0.1)",
    notas="aumento más agresivo, resto igual a línea base",
)
if resultado is None:
    print("No hay modelo guardado: corre el fit de esta sección.")
else:
    modelo_aumento, _, _, metrics_aumento = resultado

### 14.6. Experimento `sin_cw`

In [ ]:
modelo_sin_cw, metrics_sin_cw = correr_experimento(
    id_exp="sin_cw",
    cambio="sin class_weight",
    notas="mide el efecto del balanceo, resto igual a línea base",
    use_class_weight=False,
)

In [ ]:
resultado = cargar_modelos(
    "sin_cw",
    cambio="sin class_weight",
    notas="mide el efecto del balanceo, resto igual a línea base",
)
if resultado is None:
    print("No hay modelo guardado: corre el fit de esta sección.")
else:
    modelo_sin_cw, _, _, metrics_sin_cw = resultado

### 14.7. Comparación final

Ahora vamos a leer el CSV de experimentos, ordenarlo por F1 macro de mayor a menor, e imprimir las cinco columnas que importan para la comparación: id, accuracy val, F1 macro val, mejor época y descripción del cambio. La fila `linea_base` queda como referencia; los tres experimentos se leen contra ella.

In [ ]:
df_comp = pd.read_csv(TABLES_DIR / "cnn_experimentos.csv")
df_comp = df_comp.sort_values("f1_macro_val", ascending=False)
print(df_comp[["id", "accuracy_val", "f1_macro_val", "mejor_epoca", "cambio"]].to_string(index=False))

## 15. Transfer learning

Ahora vamos a empezar la fase de transfer learning: en lugar de entrenar una CNN desde cero, tomamos EfficientNetB0 preentrenada en ImageNet, la congelamos, y le agregamos una cabeza nueva de 9 clases para SylFish.

In [ ]:
base_tl = keras.applications.EfficientNetB0(  # backbone preentrenado en ImageNet
    include_top=False,                          # quitamos la capa densa de 1000 clases
    weights="imagenet",                         # cargamos los pesos preentrenados
    input_shape=(IMG_CNN, IMG_CNN, 3),          # 224x224x3, mismo tamaño que el caché
)

base_tl.trainable = False                       # congelamos todos los pesos del backbone

print(f"Entrada: {base_tl.input_shape}")
print(f"Salida:  {base_tl.output_shape}")
print(f"Params totales:     {base_tl.count_params():,}")
print(f"Params entrenables: {sum(w.shape.num_elements() for w in base_tl.trainable_weights):,}")
base_tl.summary()

### 15.1. Cabeza nueva sobre EfficientNetB0
Ahora vamos a armar el modelo completo: entrada `uint8` → aumento (el mismo de la línea base) → EfficientNetB0 congelada → GAP → Dropout → Dense(9). No ponemos `Rescaling(1/255)` porque EfficientNetB0 de Keras ya normaliza por dentro y espera [0, 255]. Llamamos al backbone con `training=False` para que sus BatchNorm queden en modo inferencia.

In [ ]:
def construir_modelo_tl(base):
    entrada = keras.Input(shape=(IMG_CNN, IMG_CNN, 3), name="entrada")   # uint8 224x224x3
    x = construir_aumento(seed=SEED)(entrada)                            # mismo aumento de la línea base
    x = base(x, training=False)                                          # BN del backbone en inferencia
    x = layers.GlobalAveragePooling2D(name="gap_tl")(x)                  # (None, 7, 7, 1280) -> (None, 1280)
    x = layers.Dropout(0.3, name="dropout_tl")(x)                        # regulariza la cabeza
    salida = layers.Dense(NUM_CLASES, activation="softmax", name="salida_tl")(x)
    return keras.Model(entrada, salida, name="cnn_sylfish_tl")

modelo_tl = construir_modelo_tl(base_tl)
modelo_tl.summary()

print(f"Salida: {modelo_tl.output_shape}")
print(f"Entrenables: {sum(w.shape.num_elements() for w in modelo_tl.trainable_weights):,}")

### 15.2. Función `entrenar_tl`
Ahora vamos a definir una función que compila, entrena con train, valida con val, y registra el experimento, igual que `correr_experimento` pero recibiendo un modelo ya armado y su tasa de aprendizaje.

In [ ]:
def entrenar_tl(modelo_tl, id_exp, lr, epochs, cambio, notas=""):
    keras.utils.set_random_seed(SEED)                                    # misma semilla que los demás experimentos
    modelo_tl.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    modelo_tl.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),                                  # nunca test
        class_weight=class_weight,
        batch_size=BATCH_SIZE,
        epochs=epochs,
        callbacks=construir_callbacks(id_exp),                           # archivos con sufijo _{id_exp}
        verbose=1,
    )
    hist_tl = pd.read_csv(TABLES_DIR / f"cnn_historial_{id_exp}.csv")
    mejor_epoca = int(hist_tl.loc[hist_tl["val_loss"].idxmin(), "epoch"])
    _, _, metrics_tl = evaluar(modelo_tl, X_val, y_val)
    registrar_experimento(id_exp, cambio, metrics_tl, mejor_epoca, notas)
    return modelo_tl, metrics_tl

### 15.3. Fase 1: backbone congelado
Ahora vamos a entrenar solo la cabeza con η = 1e-3. La celda de abajo carga el modelo desde Drive si ya existe, para no reentrenar.

In [ ]:
resultado = cargar_modelos("tl_fase1")                                   # intenta cargar de Drive
if resultado is None:                                                    # no existe: entrenamos
    modelo_tl, metrics_tl1 = entrenar_tl(
        modelo_tl, "tl_fase1", lr=1e-3, epochs=EPOCHS,
        cambio="EfficientNetB0 congelada + GAP/Dropout(0.3)/Dense(9)",
        notas="transfer learning fase 1, lr=1e-3",
    )
else:                                                                    # existe: lo cargamos y registramos
    modelo_tl, _, _, metrics_tl1 = cargar_modelos(
        "tl_fase1",
        cambio="EfficientNetB0 congelada + GAP/Dropout(0.3)/Dense(9)",
        notas="transfer learning fase 1, lr=1e-3",
    )
print(f"Fase 1 -> accuracy {metrics_tl1['accuracy']:.4f} | F1 macro {metrics_tl1['f1_macro']:.4f}")

### 15.4. Fase 2: fine-tuning
Ahora vamos a descongelar las últimas 30 capas del backbone (menos las BatchNorm) y reentrenar con η = 1e-5 por pocas épocas, partiendo de los mejores pesos de la fase 1.

In [ ]:
resultado = cargar_modelos("tl_fase2")
if resultado is None:
    base_ft = modelo_tl.get_layer(base_tl.name)                          # backbone dentro del modelo de la fase 1
    base_ft.trainable = True                                             # descongelamos todo...
    for capa in base_ft.layers[:-30]:
        capa.trainable = False                                           # ...y volvemos a congelar salvo las últimas 30
    for capa in base_ft.layers:
        if isinstance(capa, layers.BatchNormalization):
            capa.trainable = False                                       # BN sin actualizar estadísticas
    print(f"Entrenables ahora: {sum(w.shape.num_elements() for w in modelo_tl.trainable_weights):,}")
    modelo_tl, metrics_tl2 = entrenar_tl(
        modelo_tl, "tl_fase2", lr=1e-5, epochs=15,
        cambio="fine-tuning: últimas 30 capas de EfficientNetB0, lr=1e-5",
        notas="transfer learning fase 2, BN congeladas, 15 épocas máx",
    )
else:
    modelo_tl, _, _, metrics_tl2 = cargar_modelos(
        "tl_fase2",
        cambio="fine-tuning: últimas 30 capas de EfficientNetB0, lr=1e-5",
        notas="transfer learning fase 2, BN congeladas, 15 épocas máx",
    )
print(f"Fase 2 -> accuracy {metrics_tl2['accuracy']:.4f} | F1 macro {metrics_tl2['f1_macro']:.4f}")
print(pd.read_csv(TABLES_DIR / "cnn_experimentos.csv")[["id", "accuracy_val", "f1_macro_val", "mejor_epoca"]].to_string(index=False))

## 16. Selección del modelo final
Ahora vamos a elegir el experimento con mayor F1 macro en val (desempate: menor `val_loss`, luego menor tamaño). Si el ganador no es la línea base, lo copiamos como `cnn_best.keras`

In [ ]:
def ruta_best(id_exp):
    sufijo = "" if id_exp == "linea_base" else f"_{id_exp}"
    return MODELS_DIR / f"cnn_best{sufijo}.keras"

df_sel = pd.read_csv(TABLES_DIR / "cnn_experimentos.csv")
df_sel["val_loss"] = df_sel["id"].map(lambda i: pd.read_csv(TABLES_DIR / f"cnn_historial_{i}.csv")["val_loss"].min())
df_sel["tam_mb"] = df_sel["id"].map(lambda i: ruta_best(i).stat().st_size / 1e6)
df_sel = df_sel.sort_values(["f1_macro_val", "val_loss", "tam_mb"], ascending=[False, True, True])
print(df_sel[["id", "accuracy_val", "f1_macro_val", "val_loss", "tam_mb"]].to_string(index=False))

id_final = df_sel.iloc[0]["id"]
print(f"\nModelo final: {id_final}")

RUTA_FINAL = MODELS_DIR / "cnn_best.keras"                               # ruta que usan los pasos siguientes
if id_final != "linea_base":
    if (MODELS_DIR / "config_inferencia.json").exists():                 # ya hay MD5 registrado: no se toca
        RUTA_FINAL = ruta_best(id_final)
        print("config_inferencia.json existe: NO se sobrescribe cnn_best.keras. Se usa", RUTA_FINAL.name)
    else:
        shutil.copy2(MODELS_DIR / "cnn_best.keras", MODELS_DIR / "cnn_best_linea_base.keras")  # respaldo
        shutil.copy2(ruta_best(id_final), MODELS_DIR / "cnn_best.keras")
        print("cnn_best.keras reemplazado; línea base respaldada en cnn_best_linea_base.keras")

In [ ]:
# recargamos el modelo final y confirmamos las mismas cifras de la tabla
modelo_final = keras.models.load_model(RUTA_FINAL)
probs_f, preds_f, metrics_f = evaluar(modelo_final, X_val, y_val)
f1_tabla = float(df_sel.iloc[0]["f1_macro_val"])
assert abs(metrics_f["f1_macro"] - f1_tabla) < 1e-3, "El modelo recargado no reproduce el F1 de la tabla"
print(f"Accuracy val: {metrics_f['accuracy']:.4f} | F1 macro val: {metrics_f['f1_macro']:.4f}")

# métricas a JSON
metricas_json = {
    "modelo": str(id_final),
    "accuracy": float(metrics_f["accuracy"]),
    "f1_macro": float(metrics_f["f1_macro"]),
    "f1_por_clase": {c: float(metrics_f["report_dict"][c]["f1-score"]) for c in CLASES},
}
with open(TABLES_DIR / "cnn_metricas_val.json", "w") as f:
    json.dump(metricas_json, f, indent=2)

# matriz de confusión del modelo final
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(metrics_f["cm_normalized"], display_labels=CLASES).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title(f"Matriz de confusión normalizada — val ({id_final})")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_matriz_val.png", dpi=200, bbox_inches="tight")
plt.show()

## 17. Exportar predicciones de val
Ahora vamos a escribir `comparacion/pred_val_cnn.csv` con el modelo final y a verificar que coincide con `splits.csv`.

In [ ]:
COMPARACION_DIR.mkdir(parents=True, exist_ok=True)

pred_val_df = pd.DataFrame({
    "filepath": [str(p) for p in fp_val],                                # tal cual vienen del .npz
    "label": y_val,
    "pred": preds_f,                                                     # argmax del modelo final
    "conf": probs_f.max(axis=1),                                         # max softmax
})
pred_val_df.to_csv(COMPARACION_DIR / "pred_val_cnn.csv", index=False)

# verificación contra splits.csv
sub_val = splits_df[splits_df["split"] == "val"].copy()
chk = pred_val_df.assign(fp=pred_val_df["filepath"].map(_norm)).merge(
    sub_val.assign(fp=sub_val["filepath"].map(_norm))[["fp", "label"]],
    on="fp", suffixes=("", "_csv"))
assert len(pred_val_df) == len(sub_val) == len(chk), "Número de filas distinto"
assert (chk["label"] == chk["label_csv"]).all(), "label no coincide con splits.csv"
assert pred_val_df["conf"].between(0, 1).all()
print(f"pred_val_cnn.csv OK: {len(pred_val_df)} filas")

## 18. Chequeo de atajos
Ahora vamos a comprobar si la red clasifica mirando el pez o apoyándose en el fondo.

### 18.1. Descomprimir las máscaras
Ahora vamos a descomprimir `masks.zip` en `/content/masks`, que es disco local y rápido.

In [ ]:
if not (MASKS_DIR / "boal").exists():                          # solo si no se ha descomprimido ya
    MASKS_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MASKS_ZIP) as z:
        z.extractall(MASKS_DIR)                                # queda /content/masks/<clase>/<clase>_NNN.png
print(len(list(MASKS_DIR.rglob("*.png"))), "máscaras")

### 18.2. Subconjunto con máscara ok y fondo neutralizado
Ahora vamos a elegir las imágenes de val con `qc_ok = True` y a construir la versión con fondo neutralizado

In [ ]:
qc = pd.read_csv(PROCESSED_DIR / "mask_qc.csv")                # la tabla de calidad de máscaras
ok_set = set(qc.loc[qc["qc_ok"], "filepath"].map(_norm))       # rutas con máscara aprobada

idx_ok = np.array([i for i, p in enumerate(fp_val) if _norm(p) in ok_set])   # posiciones en val
y_ok = y_val[idx_ok]
conteo = np.bincount(y_ok, minlength=NUM_CLASES)
print(f"val con máscara ok: {len(idx_ok)} de {len(fp_val)}")
print(dict(zip(CLASES, conteo)))
assert (conteo > 0).all(), "Falta alguna clase en el subconjunto: evaluar() fallaría"


def neutralizar_fondo(img, ruta_mascara):
    m = cv2.imread(str(ruta_mascara), cv2.IMREAD_GRAYSCALE)   # máscara en un solo canal
    assert m is not None, f"No se pudo leer {ruta_mascara}"
    m = cv2.resize(m, (IMG_CNN, IMG_CNN), interpolation=INTER_MASK)   # a 224, sin inventar grises
    m = (m > 127) if m.max() > 1 else (m > 0)                 # booleana, sirve para 0/255 y para 0/1
    return np.where(m[..., None], img, 128).astype(np.uint8)  # pez intacto, fondo gris


X_ok = X_val[idx_ok]                                           # originales del subconjunto
X_neu = np.stack([
    neutralizar_fondo(X_val[i], MASKS_DIR / Path(_norm(fp_val[i])).with_suffix(".png"))
    for i in idx_ok
])
print(X_ok.shape, X_neu.shape, X_neu.dtype)

### 18.3. Original contra fondo neutralizado
Ahora vamos a evaluar el modelo final en las dos versiones y a guardar la tabla.

In [ ]:
_, preds_o, m_o = evaluar(modelo_final, X_ok, y_ok)            # originales
_, preds_n, m_n = evaluar(modelo_final, X_neu, y_ok)           # fondo neutralizado

atajos_df = pd.DataFrame({
    "version": ["original", "fondo_neutralizado"],
    "n": [len(idx_ok)] * 2,
    "accuracy": [m_o["accuracy"], m_n["accuracy"]],
    "f1_macro": [m_o["f1_macro"], m_n["f1_macro"]],
}).round(4)
atajos_df.to_csv(TABLES_DIR / "cnn_atajos_val.csv", index=False)
print(atajos_df.to_string(index=False))
print(f"\nCaída de accuracy: {m_o['accuracy'] - m_n['accuracy']:.4f}")
print(f"Caída de F1 macro: {m_o['f1_macro'] - m_n['f1_macro']:.4f}")

# F1 por clase: qué especies dependen más del fondo
for c in CLASES:
    print(f"{c:10s} {m_o['report_dict'][c]['f1-score']:.3f} -> {m_n['report_dict'][c]['f1-score']:.3f}")

### 18.4. Ejemplos
Ahora vamos a mostrar ocho imágenes, priorizando las que el modelo acertaba y dejó de acertar al neutralizar el fondo.

In [ ]:
cambiaron = np.where((preds_o == y_ok) & (preds_n != y_ok))[0]   # acertaba y ahora falla
resto = np.setdiff1d(np.arange(len(idx_ok)), cambiaron)          # las demás
elegidas = np.concatenate([cambiaron[:4], resto[:4 - len(cambiaron[:4])]])

fig, axes = plt.subplots(2, len(elegidas), figsize=(3 * len(elegidas), 6.5))
for j, k in enumerate(elegidas):
    axes[0, j].imshow(X_ok[k])
    axes[0, j].set_title(f"real: {CLASES[y_ok[k]]}\npred: {CLASES[preds_o[k]]}", fontsize=9)
    axes[1, j].imshow(X_neu[k])
    axes[1, j].set_title(f"pred: {CLASES[preds_n[k]]}", fontsize=9)
    axes[0, j].axis("off"); axes[1, j].axis("off")
axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("neutralizado")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_atajos_ejemplos.png", dpi=200, bbox_inches="tight")
plt.show()

## 19. Confianza
La "confianza" de la demo es $\max_k \hat{y}_k$ del softmax. No está calibrada: las redes entrenadas con entropía cruzada suelen ser sobreconfiadas, así que la leemos como preferencia relativa, no como probabilidad real.

In [ ]:
acierto = pred_val_df["pred"] == pred_val_df["label"]
print(f"Confianza media en aciertos: {pred_val_df.loc[acierto, 'conf'].mean():.3f}")
print(f"Confianza media en errores:  {pred_val_df.loc[~acierto, 'conf'].mean():.3f}")

Por ultimo vamos a verificar la consistencia de las metricas en X_test.

In [ ]:
modelo_final = keras.models.load_model(MODELS_DIR / "cnn_best.keras")   # tl_fase2
probs_t, preds_t, metrics_t = evaluar(modelo_final, X_test, y_test)     # X_test, y_test del .npz de test
print(f"Accuracy test: {metrics_t['accuracy']:.4f} | F1 macro test: {metrics_t['f1_macro']:.4f}")